# Autoencoder và VAE trên MNIST

Bạn dựng hai model cùng kiến trúc MLP: một **autoencoder** (nén rồi giải nén, tất định) và một **VAE** (encoder
ra phân phối, loss là ELBO tự viết), train trên MNIST, rồi đo cho ra số những điều phần lý thuyết nói bằng lời:
VAE đổi bao nhiêu chất lượng tái tạo để lấy khả năng **sinh**, ELBO cách log-likelihood thật bao xa (cận IWAE),
và $$\beta$$ lớn làm chiều latent "chết" (posterior collapse) thế nào.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm, lớp. Điền vào chỗ `# TODO`. Công thức KL, reparameterization, log-likelihood
  Bernoulli và cận importance-weighted là của bài Basic 01; viết lại trực tiếp bằng torch ở đây.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) đọc dữ liệu thật, train, vẽ; test bỏ qua chúng.
  Cell đầu tiên tải MNIST vào `data/raw/mnist` nếu chưa có.
- Mặc định chạy trên CPU (cả notebook khoảng 5 phút với 4 luồng). Kernel `ai-gpu` thì tự dùng GPU.
- Chấm: `uv run pytest tracks/ai-engineer/modules/06-generative-models/exercises/intermediate/01-vae-mnist -q`

In [ ]:
# explore: giới hạn luồng, tìm gốc repo, tải MNIST nếu chưa có (khoảng 11 MB, chỉ lần đầu)
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
MNIST = ROOT / "data/raw/mnist"
if not (MNIST.is_dir() and any(MNIST.iterdir())):
    sys.path.insert(0, str(ROOT))
    from data.download_dataset import ensure

    ensure("mnist")
print(MNIST, sorted(p.name for p in MNIST.iterdir()))

In [ ]:
import gzip
import math
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from torch import nn


def load_mnist(root: Path, split: str) -> tuple[torch.Tensor, torch.Tensor]:
    """Đọc tệp idx gz của MNIST (như module 01) → (X float32 (N, 784) trong [0, 1], y int64 (N,)). split: "train" | "t10k"."""
    def read_idx(path: Path) -> np.ndarray:
        data = gzip.open(path).read()
        ndim = data[3]
        shape = np.frombuffer(data, ">u4", ndim, offset=4).astype(int)
        return np.frombuffer(data, np.uint8, offset=4 + 4 * ndim).reshape(shape)

    X = read_idx(Path(root) / f"{split}-images-idx3-ubyte.gz").reshape(-1, 784).astype(np.float32) / 255
    y = read_idx(Path(root) / f"{split}-labels-idx1-ubyte.gz").astype(np.int64)
    return torch.from_numpy(X), torch.from_numpy(y)

## 1. Hai model cùng khung

Cả hai dùng MLP `784 → hidden → ... → 784`, ReLU ở tầng ẩn, decoder trả **logits** (chưa sigmoid) cho từng pixel:
likelihood là Bernoulli, $$p(x_i \mid z) = \sigma(\text{logit}_i)^{x_i}(1-\sigma(\text{logit}_i))^{1-x_i}$$, dùng được cả với pixel xám
trong $$[0, 1]$$ (mục 3 lý thuyết bàn cái giá của lựa chọn này).

- `Autoencoder(latent_dim, hidden=512)`: thuộc tính `encoder` (`784 → hidden → latent_dim`, tầng cuối không activation)
  và `decoder` (`latent_dim → hidden → 784`). `forward(x)` trả logits shape `(B, 784)`.
- `VAE(latent_dim, hidden=512)`: thuộc tính `encoder` (`784 → hidden`, ReLU), hai đầu `fc_mu` và `fc_logvar`
  (`hidden → latent_dim`), `decoder` như trên.
  - `encode(x)` trả `(mu, logvar)`, `logvar` là **log phương sai**.
  - `decode(z)` trả logits.
  - `forward(x, generator=None)` lấy một mẫu $$z = \mu + \sigma\odot\varepsilon$$ (ε từ `generator`) và trả `(logits, mu, logvar)`.

In [ ]:
class Autoencoder(nn.Module):
    """Autoencoder tất định: encoder 784 → hidden → latent_dim, decoder latent_dim → hidden → 784 (logits)."""

    def __init__(self, latent_dim: int, hidden: int = 512):
        # TODO
        raise NotImplementedError

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO
        raise NotImplementedError


class VAE(nn.Module):
    """VAE: q(z|x) = N(mu, diag(exp(logvar))), p(z) = N(0, I), p(x|z) = Bernoulli(sigmoid(decoder(z)))."""

    def __init__(self, latent_dim: int, hidden: int = 512):
        # TODO
        raise NotImplementedError

    def encode(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        # TODO
        raise NotImplementedError

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        # TODO
        raise NotImplementedError

    def forward(self, x: torch.Tensor, generator: torch.Generator | None = None) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        # TODO
        raise NotImplementedError

## 2. Loss và vòng train

- `ae_loss(model, x, generator=None)`: binary cross-entropy của tái tạo, **cộng trên 784 pixel, trung bình trên batch**
  (nats mỗi ảnh). `generator` không dùng, có mặt để cùng chữ ký.
- `vae_loss(model, x, beta=1.0, generator=None)`: trả `(loss, recon, kl)`, ba số vô hướng: `recon` là trung bình theo
  batch của $$-\log p(x \mid z)$$ (nats mỗi ảnh), `kl` là trung bình của $$\mathrm{KL}(q(z \mid x)\,\|\,p(z))$$ dạng đóng, và
  `loss = recon + beta * kl` (tức $$-\text{ELBO}$$ khi $$\beta = 1$$).
- `train(model, loss_fn, X, epochs, lr=1e-3, batch_size=128, seed=0, device="cpu")`: Adam, xáo dữ liệu mỗi epoch, mọi số
  ngẫu nhiên (thứ tự, ε) từ một `torch.Generator` seed `seed` (trên CPU). `loss_fn(model, xb, generator)` trả loss vô hướng
  (với VAE, bọc `vae_loss` trong một `lambda` lấy phần tử đầu). Trả list loss trung bình **theo mẫu** của từng epoch;
  khi xong model ở `eval()`.

In [ ]:
def ae_loss(model: nn.Module, x: torch.Tensor, generator: torch.Generator | None = None) -> torch.Tensor:
    """BCE tái tạo cộng trên pixel, trung bình theo batch."""
    # TODO
    raise NotImplementedError


def vae_loss(model: VAE, x: torch.Tensor, beta: float = 1.0, generator: torch.Generator | None = None) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """(recon + beta * kl, recon, kl), mỗi cái là trung bình theo batch của đại lượng theo mẫu (nats)."""
    # TODO
    raise NotImplementedError


def train(model: nn.Module, loss_fn, X: torch.Tensor, epochs: int, lr: float = 1e-3, batch_size: int = 128,
          seed: int = 0, device: str = "cpu") -> list[float]:
    """Adam; mỗi epoch xáo X bằng generator seed `seed`; trả loss trung bình theo mẫu mỗi epoch; kết thúc ở eval()."""
    # TODO
    raise NotImplementedError

## 3. Đo cho đúng: cận IWAE và số chiều latent còn sống

ELBO chỉ là **cận dưới** của $$\log p(x)$$. Để biết model tốt thật bao nhiêu, ước lượng log-likelihood bằng importance
sampling với $$K$$ mẫu từ $$q(z \mid x)$$ (Burda et al., 2016):

$$
\log p(x) \gtrsim \mathcal{L}_K = \log \frac{1}{K}\sum_{k=1}^{K} \frac{p(x \mid z_k)\,p(z_k)}{q(z_k \mid x)}, \qquad z_k \sim q(z \mid x)
$$

$$\mathcal{L}_1$$ là ELBO, $$\mathcal{L}_K$$ tăng theo $$K$$ và tiến tới $$\log p(x)$$.

- `iwae_nll(model, X, K=100, batch_size=500, seed=0)`: trung bình của $$-\mathcal{L}_K$$ trên các ảnh của `X` (nats mỗi
  ảnh). ε lấy từ `torch.Generator` seed `seed`; tính trong `torch.no_grad()`, ổn định số (log-sum-exp). Chạy trên device
  của model.
- `active_units(model, X, threshold=0.01)`: số chiều latent $$j$$ có $$\mathrm{Var}_x\big(\mu_j(x)\big) > $$ `threshold`, phương
  sai (chia $$N$$) của mean encoder trên các ảnh `X` (Burda et al.). Một chiều mà $$\mu_j$$ gần như không đổi theo ảnh là
  chiều decoder không dùng tới.

In [ ]:
@torch.no_grad()
def iwae_nll(model: VAE, X: torch.Tensor, K: int = 100, batch_size: int = 500, seed: int = 0) -> float:
    """Trung bình -L_K (nats mỗi ảnh) với K mẫu z từ q(z|x)."""
    # TODO
    raise NotImplementedError


@torch.no_grad()
def active_units(model: VAE, X: torch.Tensor, threshold: float = 0.01) -> int:
    """Số chiều j có Var_x(mu_j(x)) > threshold."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: đọc MNIST thật, chọn device và cấu hình
X_train, y_train = load_mnist(MNIST, "train")
X_test, y_test = load_mnist(MNIST, "t10k")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EPOCHS = 20
print(X_train.shape, X_test.shape, DEVICE, "| mean pixel", round(X_train.mean().item(), 4),
      "| tỉ lệ pixel đúng 0 hoặc 1:", round(((X_train == 0) | (X_train == 1)).float().mean().item(), 3))

In [ ]:
# explore: train AE và VAE cùng latent 16, cùng ngân sách 20 epoch (khoảng 1 phút mỗi model trên CPU)
torch.manual_seed(0)
results = {}
def run(name, model, loss_fn):
    t0 = time.perf_counter()
    hist = train(model, loss_fn, X_train, EPOCHS, device=DEVICE)
    results[name] = model
    print(f"{name:12s} loss epoch 1 {hist[0]:.2f}, cuối {hist[-1]:.2f} | {time.perf_counter() - t0:.0f} s", flush=True)

torch.manual_seed(0)
run("AE-16", Autoencoder(16), ae_loss)
torch.manual_seed(0)
run("VAE-16", VAE(16), lambda m, x, g: vae_loss(m, x, 1.0, g)[0])

In [ ]:
# explore: đo trên 10 000 ảnh test. AE: BCE tái tạo. VAE: recon, KL, -ELBO (một mẫu z), NLL ước lượng IWAE K = 1 và 100
ae, vae = results["AE-16"], results["VAE-16"]
with torch.no_grad():
    ae_rec = ae_loss(ae, X_test.to(DEVICE)).item()
    g = torch.Generator(device=DEVICE).manual_seed(0)
    neg_elbo, rec, kl = (v.item() for v in vae_loss(vae, X_test.to(DEVICE), 1.0, g))
print(f"AE-16 : BCE tái tạo {ae_rec:.1f} nats/ảnh")
print(f"VAE-16: recon {rec:.1f} + KL {kl:.1f} = -ELBO {neg_elbo:.1f} nats/ảnh")
nll1 = iwae_nll(vae, X_test, K=1)
nll100 = iwae_nll(vae, X_test, K=100)
print(f"VAE-16: -L_1 {nll1:.1f}, -L_100 {nll100:.1f} nats/ảnh, khoảng hở ELBO - log p(x) ước lượng {nll1 - nll100:.1f} nats")
print(f"VAE-16: active units {active_units(vae, X_test)}/16 | bits/dim (-L_100) {nll100 / (784 * math.log(2)):.3f}")

In [ ]:
# explore: classifier MLP nhỏ (2 epoch, chỉ để chấm mẫu sinh): ảnh "giống chữ số" thì classifier tự tin
torch.manual_seed(0)
clf = nn.Sequential(nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, 10)).to(DEVICE)
opt = torch.optim.Adam(clf.parameters(), 1e-3)
g = torch.Generator().manual_seed(0)
for _ in range(2):
    perm = torch.randperm(len(X_train), generator=g)
    for i in range(0, len(X_train), 128):
        b = perm[i:i + 128]
        loss = F.cross_entropy(clf(X_train[b].to(DEVICE)), y_train[b].to(DEVICE))
        opt.zero_grad(); loss.backward(); opt.step()
clf.eval()
with torch.no_grad():
    print("classifier test acc:", (clf(X_test.to(DEVICE)).argmax(1).cpu() == y_test).float().mean().item())

@torch.no_grad()
def judge(imgs):
    """(tỉ lệ ảnh classifier tự tin > 0.9, số lớp xuất hiện, histogram lớp dự đoán)."""
    p = clf(imgs.to(DEVICE)).softmax(1).cpu()
    pred = p.argmax(1)
    return (p.max(1).values > 0.9).float().mean().item(), int(torch.bincount(pred, minlength=10).gt(0).sum()), torch.bincount(pred, minlength=10).tolist()

In [ ]:
# explore: SINH ảnh mới: lấy z ~ N(0, I) rồi decode, cho cả AE và VAE. Ảnh thật làm mốc.
import matplotlib.pyplot as plt

gz = torch.Generator().manual_seed(1)
z = torch.randn(1000, 16, generator=gz).to(DEVICE)
with torch.no_grad():
    ae_codes = ae.encoder(X_test.to(DEVICE))
    samples = {"ảnh thật (test)": X_test[:1000],
               "AE, z ~ N(0, I)": torch.sigmoid(ae.decoder(z)).cpu(),
               "VAE, z ~ N(0, I)": torch.sigmoid(vae.decode(z)).cpu()}
print("std mã AE theo từng chiều (min / max):", round(ae_codes.std(0).min().item(), 2), "/", round(ae_codes.std(0).max().item(), 2))
fig, axes = plt.subplots(3, 10, figsize=(12, 4))
for row, (name, imgs) in enumerate(samples.items()):
    conf, n_cls, hist = judge(imgs)
    print(f"{name:18s}: tự tin > 0.9: {conf:.3f} | số lớp: {n_cls} | histogram: {hist}")
    for j in range(10):
        axes[row, j].imshow(imgs[j].reshape(28, 28), cmap="gray"); axes[row, j].axis("off")
    axes[row, 0].set_title(name, fontsize=8, loc="left")
plt.show()

In [ ]:
# explore: beta và posterior collapse: VAE-16 với beta = 0.5, 4, 10 (cùng 20 epoch); đo recon, KL, active units
rows = [("beta 1", vae)]
for beta in (0.5, 4.0, 10.0):
    torch.manual_seed(0)
    m = VAE(16)
    train(m, lambda mm, x, g, b=beta: vae_loss(mm, x, b, g)[0], X_train, EPOCHS, device=DEVICE)
    rows.append((f"beta {beta:g}", m))
for name, m in rows:
    with torch.no_grad():
        _, rec, kl = (v.item() for v in vae_loss(m, X_test.to(DEVICE), 1.0, torch.Generator(device=DEVICE).manual_seed(0)))
        conf, n_cls, _ = judge(torch.sigmoid(m.decode(z)).cpu())
    print(f"{name:9s} recon {rec:6.1f} | KL {kl:5.1f} | -ELBO {rec + kl:6.1f} | active units {active_units(m, X_test):2d}/16 | mẫu sinh tự tin > 0.9: {conf:.3f}")

In [ ]:
# explore: VAE latent 2 chiều: bản đồ mean encoder của ảnh test (màu = nhãn) và lưới decode
torch.manual_seed(0)
vae2 = VAE(2)
train(vae2, lambda m, x, g: vae_loss(m, x, 1.0, g)[0], X_train, EPOCHS, device=DEVICE)
with torch.no_grad():
    mu2 = vae2.encode(X_test.to(DEVICE))[0].cpu()
    grid = torch.stack(torch.meshgrid(torch.linspace(-2.5, 2.5, 15), torch.linspace(-2.5, 2.5, 15), indexing="xy"), -1).reshape(-1, 2)
    dec = torch.sigmoid(vae2.decode(grid.flip(0).to(DEVICE))).cpu()
    _, rec2, kl2 = (v.item() for v in vae_loss(vae2, X_test.to(DEVICE), 1.0, torch.Generator(device=DEVICE).manual_seed(0)))
print(f"VAE-2: recon {rec2:.1f}, KL {kl2:.1f}, -ELBO {rec2 + kl2:.1f}")
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 5.5))
sc = a1.scatter(mu2[:, 0], mu2[:, 1], c=y_test, cmap="tab10", s=2)
a1.legend(*sc.legend_elements(), fontsize=7, loc="upper right"); a1.set_title("mean encoder của 10 000 ảnh test")
canvas = dec.reshape(15, 15, 28, 28).permute(0, 2, 1, 3).reshape(15 * 28, 15 * 28)
a2.imshow(canvas, cmap="gray"); a2.axis("off"); a2.set_title("decode lưới z trong [-2.5, 2.5]^2")
plt.show()

In [ ]:
# explore: nội suy trong latent giữa hai ảnh test: AE (đường thẳng giữa hai mã) và VAE (giữa hai mean)
a, b = X_test[3:4].to(DEVICE), X_test[2:3].to(DEVICE)
ts = torch.linspace(0, 1, 10, device=DEVICE)[:, None]
with torch.no_grad():
    za, zb = ae.encoder(a), ae.encoder(b)
    ae_path = torch.sigmoid(ae.decoder((1 - ts) * za + ts * zb)).cpu()
    ma, mb = vae.encode(a)[0], vae.encode(b)[0]
    vae_path = torch.sigmoid(vae.decode((1 - ts) * ma + ts * mb)).cpu()
fig, axes = plt.subplots(2, 10, figsize=(12, 2.6))
for j in range(10):
    axes[0, j].imshow(ae_path[j].reshape(28, 28), cmap="gray"); axes[0, j].axis("off")
    axes[1, j].imshow(vae_path[j].reshape(28, 28), cmap="gray"); axes[1, j].axis("off")
axes[0, 0].set_title("AE", fontsize=8, loc="left"); axes[1, 0].set_title("VAE", fontsize=8, loc="left")
plt.show()

## 4. Đọc kết quả

Trước khi làm trắc nghiệm trong app, tự trả lời:

1. AE tái tạo tốt hơn VAE bao nhiêu nats mỗi ảnh? Vì sao vậy mà mẫu sinh từ $$z \sim \mathcal{N}(0, I)$$ của AE lại tệ hơn hẳn?
   Gợi ý: nhìn std mã AE theo từng chiều.
2. Khoảng hở giữa $$-\mathcal{L}_1$$ và $$-\mathcal{L}_{100}$$ nói gì về posterior xấp xỉ $$q(z \mid x)$$?
3. Khi $$\beta$$ tăng, KL và số active units đổi thế nào? Chất lượng mẫu sinh có tốt lên không? Đó có phải "posterior
   collapse" theo nghĩa xấu không?